In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

### Load Data

In [2]:
# import data from csv 
file_path_1 = os.path.join('..', 'data', 'raw', 'mimic', 'D_ICD_DIAGNOSES.csv')
df_icd = pd.read_csv(file_path_1)
print(df_icd.shape)
df_icd.head()

(14567, 4)


,ROW_ID,ICD9_CODE,SHORT_TITLE,LONG_TITLE
0,174,01166,TB pneumonia-oth test,"Tuberculous pneumonia [any form], tubercle bac..."
1,175,01170,TB pneumothorax-unspec,"Tuberculous pneumothorax, unspecified"
2,176,01171,TB pneumothorax-no exam,"Tuberculous pneumothorax, bacteriological or h..."
3,177,01172,TB pneumothorx-exam unkn,"Tuberculous pneumothorax, bacteriological or h..."
4,178,01173,TB pneumothorax-micro dx,"Tuberculous pneumothorax, tubercle bacilli fou..."


In [3]:
file_path_2 = os.path.join('..', 'data', 'raw', 'mimic', 'DIAGNOSES_ICD.csv')
df_diagnoses = pd.read_csv(file_path_2)
print(df_diagnoses.shape)
df_diagnoses.head()

(651047, 5)


,ROW_ID,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE
0,1297,109,172335,1.0,40301
1,1298,109,172335,2.0,486
2,1299,109,172335,3.0,58281
3,1300,109,172335,4.0,5855
4,1301,109,172335,5.0,4254


In [4]:
# how often does each ICD9 code appear in the dataset?
df_diagnoses['ICD9_CODE'].value_counts()
# only keep the top 50 most common ICD9 codes
top_50 = df_diagnoses['ICD9_CODE'].value_counts().index[:50]
df_diagnoses = df_diagnoses[df_diagnoses['ICD9_CODE'].isin(top_50)]
df_diagnoses.shape


(242728, 5)

In [5]:
# match the ICD9 codes to their descriptions and drop one row_ID column
df_diagnoses = df_diagnoses.merge(df_icd, left_on='ICD9_CODE', right_on='ICD9_CODE')
df_diagnoses = df_diagnoses.drop(columns=['ROW_ID_x'])
df_diagnoses.head()

,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE,ROW_ID_y,SHORT_TITLE,LONG_TITLE
0,109,172335,2.0,486,5528,"Pneumonia, organism NOS","Pneumonia, organism unspecified"
1,117,164853,11.0,486,5528,"Pneumonia, organism NOS","Pneumonia, organism unspecified"
2,124,112906,2.0,486,5528,"Pneumonia, organism NOS","Pneumonia, organism unspecified"
3,124,138376,8.0,486,5528,"Pneumonia, organism NOS","Pneumonia, organism unspecified"
4,136,184644,2.0,486,5528,"Pneumonia, organism NOS","Pneumonia, organism unspecified"


In [6]:
df_diagnoses.dtypes

SUBJECT_ID       int64
HADM_ID          int64
SEQ_NUM        float64
ICD9_CODE       object
ROW_ID_y         int64
SHORT_TITLE     object
LONG_TITLE      object
dtype: object

In [7]:
# unique patients
df_diagnoses['SUBJECT_ID'].nunique()

43633

In [8]:
# unique encounters
df_diagnoses['HADM_ID'].nunique()

55078

In [9]:
file_path_3 = os.path.join('..', 'data', 'raw', 'mimic', 'NOTEEVENTS.csv')
df_text = pd.read_csv(file_path_3)
print(df_text.shape)

/var/folders/7p/dm1d3s0909bb57crg9cl5z3m0000gn/T/ipykernel_30499/4130333595.py:2: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_text = pd.read_csv(file_path_3)


(2083180, 11)


In [10]:
# HADM_ID to int
df_text['HADM_ID'] = df_text['HADM_ID'].astype('Int64')
df_text.head()

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT
0,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...
1,175,13702,107527,2118-06-14,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-6-2**] Discharg...
2,176,13702,167118,2119-05-25,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2119-5-4**] D...
3,177,13702,196489,2124-08-18,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2124-7-21**] ...
4,178,26880,135453,2162-03-25,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2162-3-3**] D...


In [11]:
df_text.dtypes

ROW_ID           int64
SUBJECT_ID       int64
HADM_ID          Int64
CHARTDATE       object
CHARTTIME       object
STORETIME       object
CATEGORY        object
DESCRIPTION     object
CGID           float64
ISERROR        float64
TEXT            object
dtype: object

In [12]:
# how many nan values are there in each column?
df_text.isna().sum()

ROW_ID               0
SUBJECT_ID           0
HADM_ID         231836
CHARTDATE            0
CHARTTIME       316566
STORETIME       836776
CATEGORY             0
DESCRIPTION          0
CGID            836776
ISERROR        2082294
TEXT                 0
dtype: int64

In [13]:
# how to match the text data to the diagnoses data?